# Installing packages

In [1]:
!pip install torch torchtext sentencepiece nltk bert-score pandas numpy matplotlib

# Loading Dataset

In [2]:
import pandas as pd

def dataset_load(sa_path, en_path):
    sa_df = pd.read_csv(sa_path)
    en_df = pd.read_csv(en_path)
    merged_df = pd.merge(sa_df, en_df, on="Source_id")
    merged_df.rename(columns={"Sentence_sa": "source_lang", "Sentence_en": "target_lang"}, inplace=True)
    return merged_df[["Source_id", "source_lang", "target_lang"]]

train_df = dataset_load("data/train_sa_10000.csv", "data/train_en_10000.csv")
dev_df = dataset_load("data/dev_sa_1000.csv", "data/dev_en_1000.csv")
test_df = dataset_load("data/test_sa_1000.csv", "data/test_en_1000.csv")


# SentencePiece Tokenizer training

In [3]:
import sentencepiece as sent_pi

def train_sentence_piece(sentences, model_prefix, vocab_size=8000):
  with open(f"{model_prefix}.txt", "w", encoding="utf-8") as f:
    for sentence in sentences:
      f.write(sentence + "\n")

  sent_pi.SentencePieceTrainer.train(
    input=f"{model_prefix}.txt",
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    character_coverage=1.0,
    model_type="bpe",
  )

## Preparing Sentences


In [4]:
source_sentences = train_df["source_lang"].tolist()
target_sentences = train_df["target_lang"].tolist()

train_sentence_piece(source_sentences, "spm_source", vocab_size=8000)
train_sentence_piece(target_sentences, "spm_target", vocab_size=8000)

## Load Tokenizers

In [5]:
source_sp = sent_pi.SentencePieceProcessor()
source_sp.load("spm_source.model")
target_sp = sent_pi.SentencePieceProcessor()
target_sp.load("spm_target.model")

True

# Dataset Loader and Dataset

## Importing packages

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

## Translation Dataset

In [11]:
class TranslationDataset(Dataset):
    def __init__(self, data_frame, source_sp, target_sp, max_length=128):
        self.data_frame = data_frame
        self.source_sp = source_sp
        self.target_sp = target_sp
        self.max_length = max_length
        self.pad_index = 0

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, index):
        source_text = self.data_frame.iloc[index]["source_lang"]
        target_text = self.data_frame.iloc[index]["target_lang"]
        source_indexes = [self.source_sp.bos_id()] + self.source_sp.EncodeAsIds(source_text) + [self.source_sp.eos_id()]
        target_indexes = [self.target_sp.bos_id()] + self.target_sp.EncodeAsIds(target_text) + [self.target_sp.eos_id()]
        source_indexes = source_indexes[:self.max_length]
        target_indexes = target_indexes[:self.max_length]
        return torch.tensor(source_indexes, dtype=torch.long), torch.tensor(target_indexes, dtype=torch.long)

## Creating a collate function

In [12]:
def collate(batch):
  source_batch, target_batch = zip(*batch)
  source_lengths = [len(s) for s in source_batch]
  target_lengths = [len(t) for t in target_batch]
  source_padded = pad_sequence(source_batch, batch_first=True, padding_value=0)
  target_padded = pad_sequence(target_batch, batch_first=True, padding_value=0)
  return source_padded, source_lengths, target_padded, target_lengths

## Using TranslationDataset Class

In [13]:
train_dataset = TranslationDataset(train_df, source_sp, target_sp)
dev_dataset = TranslationDataset(dev_df, source_sp, target_sp)
test_dataset = TranslationDataset(test_df, source_sp, target_sp)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate)

# Creating a Transformer Model
Here we will create transformer model. We will be using following details for this purpose:-


*   Embedding Layer
*   Positional Encodeing
*   Multi-head Attention
*   Feed-forward networks
*   Encoder and Decoder Stacks
*   Final linear layer with softmax




## Positional Encoding

We will be using Sinusoidal positional encoding. It is a technique to inject information about sequence order of tokens